# Practice Session 03: Find near-duplicates using shingling

In this session we will take a large corpus of tweets and detect near-duplicates on this corpus using a technique known as *shingling*.

Two documents are considered near-duplicates if they share a large amount of ngrams. The *ngrams* of a phrase are overlapping sequences of words of length *n*. For instance, the phrase '*May your vibes be high, your boundaries be thick, and your peace of mind be absolutely non-negotiable.*' has the following 3-grams:

* 'may your vibes'
* 'your vibes be'
* 'vibes be high'
* 'be high your'
* ...
* 'mind be absolutely'
* 'be absolutely non-negotiable'

To measure the similarity between two sets, we will use the [Jaccard index](https://en.wikipedia.org/wiki/Jaccard_index), which is the size of the intersection of the two sets divided by their union. This values goes between 0.0 (meaning the documents have no ngrams in common) to 1.0 (meaning the documents have the same ngrams).

To speed up things, instead of comparing the set of shingles of two documents which can be large, we will derive a fixed-length *signature* or *sketch* for each document. This will be obtained by (1) applying a random permutation to the list of possible ngrams, and (2) pick the ngram that appears first in the permuted list. The Jaccard index between these signatures will be a good approximation of the Jaccard index between the original sets of ngrams. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

⭐ **Regarding the usage of generative AI.** You may use generative AI to (a) understand how to do certain operations, and (b) generate short snippets of code, that you can incorporate into your practice after reviewing them. In that case, add a comment `#AI` before the snippet. You can also add `#AI (edited)` if you edited the code before incorporating it.

⚠️ It is forbidden to use generative AI in these practices for answering the text-based questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

Author: <font color="#2680EB">Your name here</font>

E-mail: <font color="#2680EB">Your e-mail here</font>

AI used: <font color="#2680EB">Maker and model, or "none".</font>

Date: <font color="#2680EB">The current date here</font>

# 0. Dataset

The corpus you will use contains about [400,000 pairs of questions posted in Quora](https://www.kaggle.com/datasets/quora/question-pairs-dataset). 

Then, the file is compressed using `gzip`, and we will read it in compressed form.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [1]:
# LEAVE AS-IS

# Required libraries
import gzip
import csv
import re
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Romanization library
from transliterate import translit
from transliterate.exceptions import LanguageDetectionError

# Timing
from timeit import default_timer as timer


The input consists of pairs and an indicator (0,1) of whether both questions are semantically equivalent.

We will keep "long" questions, specifically, only read a question when both questions in a pair have at least 20 words.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [2]:
# LEAVE AS-IS

# Input file
INPUT_FILENAME = "questions.csv.gz"

# Minimum number of words needed to read the questions
MIN_WORDS = 20

# IMPORTANT:
# 1. Set this to 2000 during development
# 2. Set this to 10000 once you have completed the development
MAX_QUESTIONS = 2000

# Array for storing questions
questions = []

with gzip.open(INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
    
    questions_read = 0
    reader = csv.DictReader(input_file)
    headers = reader.fieldnames
    
    for line in reader:
            
        # Read line
        q1 = line["question1"]
        q2 = line["question2"]
        
        if min(len(q1.split()), len(q2.split())) >= MIN_WORDS:
            questions.append(q1)
            questions.append(q2)
            
            questions_read += 2
        
            if (MAX_QUESTIONS != -1) and (questions_read >= MAX_QUESTIONS):
                break

print(f"Read {len(questions)} questions")

NameError: name 'gzip' is not defined

The function [`random.sample`](https://docs.python.org/3/library/random.html#random.sample) can be used to obtain a random sample from a list.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [3]:
# LEAVE AS-IS

SAMPLE_SIZE = 3

for question in random.sample(questions, SAMPLE_SIZE):
    print(question)
    print()

I'm jealous when my best friend is having fun with other friends.That makes me angry with her and sometimes I don't talk to her because of it. As a girl, how can I stop being so jealous?

Just completed my B.Tech with an aggregate of 66%, X and XII 75, 57 respectively. The eligibilty criteria in most of companies is 60% and above in X, XII and graduation. Now that I am helpless and cannot do anything about my XII score, does this mean that I wont get a job anywhere?

What is the best answer given by a person, when there are too many good morning messages in a WhatsApp group?



# 1. Auxiliary functions <font color="#2680EB">[5%]</font>

Implement the Jaccard similarity between two lists: the size of the intersection of two sets, divided by the size of their union.

You can use set operations: `set(l)` to convert a list `l` to a set, then `set1.union(set2)` and `set1.intersection(set2)` to compute union and intersection of sets `set1`, `set2`. Learn more in this [tutorial on set operations](https://learnpython.com/blog/python-set-operations/)

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Indicate how would you handle the case in which both lists are empty.** Include a justification, ideally a reference, justifying your choice.</font>

**Answer.** If both lists are empty:

* YOUR ANSWER HERE


<font size="+1" color="#2680EB">**Implement Jaccard similarity of lists.** Be consistent with your choice of handling of empty lists.</font>

In [ ]:
def jaccard_similarity(list1, list2):
    
    # YOUR CODE HERE
    
    return sim

Write code to test your function. Your tests cases should be:

1. Handling of empty lists
1. Two lists for which the jaccard similarity is 1.0
1. Two lists for which the jaccard similarity is 1/6 (+- 0.001)

Use the `assert` function for this test.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Test your Jaccard similarity function,** with at least the test cases described above.</font>

In [ ]:
# YOUR CODE HERE

Implement a function `clean` that cleans-up questions according to this specification:

1. Converting to lowercase
1. [Romanizing](https://en.wikipedia.org/wiki/Romanization) text, replacing "Ñ" by "n", "ñ" by "n", "ó" by "o", etc, using the [transliterate](https://pypi.org/project/transliterate/) library. 
1. Removing spaces at the beginning and spaces at the end with the `strip()` function.
1. Removing anything that remains that is not a letter or digit
1. Changing double spaces to single spaces.

You can use `text.lower()` to convert to lowercase, and then `re.sub(...)` to replace parts of the text. See [Python regexps](https://docs.python.org/3/library/re.html).

Note that `translit` may throw a `LanguageDetectionError`. In that case, simply assume that the text does not contain any non-ascii character.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement your text cleaning function**</font>

In [ ]:
def clean(text):
    text = text.lower()
    
    # YOUR CODE HERE
    return text

Test your function by passing it five different texts including punctuation, non-Roman characters, URLs, etc. Make sure your test cases cover all the aspects of the specification (at least one test case per point above).

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Test your text cleaning function.** Use [assert](https://realpython.com/python-assert-statement/), and cover all cases of the specification. Make sure your test cases are similar to questions in the dataset.</font>

In [ ]:
# YOUR CODE HERE

# 2. Implement an n-gram extractor <font color="#2680EB">[5%]</font>

Implement the function `ngrams(text,size)`, which should produce all sub-sequences of `size` words present in the cleaned text. 

Note that `ngrams` is a list, and each element of that list is a *string*.

The only words you must consider in a ngram are words having at least `MIN_TOKEN_LENGTH` characters.

You can use the [split](https://docs.python.org/2/library/string.html#string.split) and [join](https://docs.python.org/2/library/string.html#string.join) function of the split library. Remember that to extract elements *i* to *j* of array *a* you use `a[i:j]`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement the ngram extractor.**</font>

In [ ]:
MIN_TOKEN_LENGTH = 4

def ngrams(text, size):
    
    # YOUR CODE HERE 
    
    return(ngrams)

Test your function with ngrams of size 3.

Remember that `ngrams` should return a list of string, not a list of lists, so carefully check that you are returning a list of strings and not a list of lists.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Test your function 'ngrams'**. Consider ngrams of size 3, select three questions from the dataset, and check their ngrams using assert.</font>

In [ ]:
# YOUR CODE HERE

# 3. Estimation of brute force all-pairs method  <font color="#2680EB">[10%]</font>

The following code, which you should leave as-is, computes the time in seconds that it takes to compare all first *limit* messages against all first *limit* messages in the array.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [10]:
# LEAVE AS-IS

def time_brute_force_similarities(texts, limit, ngram_size):
    if limit > len(texts):
        raise ValueError("Limit should be less than or equal than the number of texts")
        
    # Start a timer
    start = timer()

    # Iterate through document identifiers
    for docid1 in range(np.min([len(texts), limit])):

        # Extract ngrams from doc1
        doc1 = texts[docid1]
        ngrams1 = ngrams(doc1, ngram_size)

        # Iterate through document identifiers larger than doc2
        for docid2 in range(docid1+1, np.min([len(questions), limit])):
                         
            # Extract ngrams from doc2
            doc2 = texts[docid2]
            ngrams2 = ngrams(doc2, ngram_size)

            # Compute similarity
            similarity = jaccard_similarity(ngrams1, ngrams2)

    end = timer()
    return(end - start)

Use the function above to create a plot in which you have in the x axis the number of messages to check, and in the y axis the time it takes to check that many messages if we use ngrams of size **3**. Try with x from *1* to *1001* in increments of *200* (use the [range](https://docs.python.org/3/library/functions.html#func-range) function).

In all plot, remember to label title, axis X, and axis Y. In this case you can use `plt.title()`, `plt.xlabel()`, and `plt.ylabel()`. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Generate the data for the requested plots**.</font>

In [1]:
num_texts = []
time_required = []

# YOUR CODE HERE  


In [ ]:
# Now plot this data; remember to add a title and label for the x and y axis.

# YOUR CODE HERE

Next, fit by trial and error a quadratic curve `y = a * (x**2.0)` to what you see in the plot. For instance (you will need to modify the coefficient below to make this fit):

```python
time_estimated = [(0.0002*(n**2.0)) for n in num_texts]
...
sns.lineplot(x=num_texts, y=time_required, label="Experimental")
sns.lineplot(x=num_texts, y=time_estimated, label="Estimation")
```

Remember to include titles for the plot and the axes and the [legend](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html) of the plot.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Find a quadratic approximation of the required time for bruteforce**. Use two significant digits for the coefficient. The code will plot your estimation and the experimental value.</font>

In [ ]:
# YOUR CODE HERE

sns.lineplot(x=num_texts, y=time_required, label="Experimental")
sns.lineplot(x=num_texts, y=time_estimated, label="Estimation")

plt.xlabel('Number of messages')
plt.ylabel('Time [s]')
_ = plt.title('Required time')

<font size="+1" color="#2680EB">**Estimate how long the brute-force approach would take for the entire dataset**. It contains 25K questions. Express your estimation first in seconds, and then in human terms (e.g., hours, or days) using [timedelta](https://docs.python.org/3/library/datetime.html#timedelta-objects) and [humanize](https://pypi.org/project/humanize/).</font>

In [ ]:
from datetime import timedelta
import humanize

# YOUR CODE HERE

# 4. Computing the doc-ngram matrix <font color="#2680EB">[20%]</font>

Now we will compute a matrix in which every row is an ngram, and every column is a document.

In real-world implementations, this is done by hashing the ngrams and then every row is an ngram *hash*; in this practice we will skip that step and work directly with one ngram per row, which is conceptually the same and easier to code.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

## 4.1 Create list of all ngrams

Implement code to create:

* the dictionary `ngram_to_index`, which should convert an ngram to an index (a row number),
* the dictionary `index_to_ngram`, which should convert an index to an ngram, and
* the variable `num_distinct_ngrams` which should contain the number of distinct ngrams.

Note that the total number of n-grams may vary depending on how you `clean()` text.

In this dataset it should be about 10 times the number of documents.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Create an ngram_to_index dictionary**.</font>

In [ ]:
NGRAM_SIZE = 3

ngram_to_index = {}
index_to_ngram = {}
next_index = 0

for question in questions:
    all_ngrams = ngrams(question, NGRAM_SIZE)
    
    # YOUR CODE HERE
            
num_distinct_ngrams = next_index

print(f"There are {num_distinct_ngrams} distinct ngrams in the {len(questions)} questions")

There are 20065 distinct ngrams in the 2000 questions


Test your function by checking consistency.

Note that the exact index varies, depending on how you `clean()` text. What is important is that when you print the `index_to_ngram` of the returned index, it should give you the same string. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Write test_ngram_index_consistency**.</font>

In [ ]:
def test_ngram_index_consistency(ngram_to_index, index_to_ngram, text):
    # YOUR CODE HERE

test_ngram_index_consistency(ngram_to_index, index_to_ngram, "pending many days")
test_ngram_index_consistency(ngram_to_index, index_to_ngram, "answer this question")


Ngram 'pending many days' has index 6954, checked correctly
Ngram 'answer this question' has index 7778, checked correctly


## 4.2 Create table ngrams x documents

Now we will create a boolean matrix named `M_ngram_doc`, where each row should be an n-gram, and each column should be a document.

There might be documents having less than *NGRAM_SIZE* words and thus containing no shingles. You can skip those documents above (when reading the file), or handle them here.

The next code creates a matrix containing `False` in all cells. Leave as-is. If you run out of memory, limit the number of documents you read at the beginning of this file, for instance, read only the first 10,000 or the first 7,000 documents, and then try again.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [14]:
# LEAVE AS-IS

# Create dense matrix in which every cell contains the value "False"
M_ngram_doc = np.full((num_distinct_ngrams, len(questions)), False)

# Print the number of rows and columns of this matrix
# numpy.matrix.shape is a tuple (0=rows, 1=columns)
print(f"Matrix has {M_ngram_doc.shape[0]} rows (distinct shingles)")
print(f"Matrix has {M_ngram_doc.shape[1]} columns (distinct documents)")

Matrix has 20065 rows (distinct shingles)
Matrix has 2000 columns (distinct documents)


Complete the matrix `M_ngram_doc` so that position i, j (row, column) holds a `True` if document j contains ngram i, otherwise it will have the default value we created above, `False`.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Fill-in the M_ngram_doc matrix**. Use the template below.</font>

In [ ]:
for docid in range(len(questions)):
    question = questions[docid]
    all_ngrams = ngrams(question, NGRAM_SIZE)
    for ngram_str in all_ngrams:
        # YOUR CODE HERE

Measure the density of this matrix, as a percentage. This is the number of non-zeroes in the matrix as a percentage of the number of cells of the matrix.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Print the density of the matrix**. This is the fraction of cells that are non-zero. Print this as a percentage, using 3 decimals.</font>

In [ ]:
# YOUR CODE HERE

Density: 0.060%


Next, we print one document (column). All columns should be very sparse, i.e., mostly zeroes.

Note that the specific ngram ids you will get depend on your cleanup process, and that the output is in ascending order of ngram number, not in the same ordering in which the ngrams appear in the message.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [17]:
# LEAVE AS-IS

test_docid=391
print(f"Positions of non-zeros in column of docid {test_docid} of M_ngram_doc")
print()

print(f"Clean text:\n {clean(questions[test_docid])}")
ngrams_doc_tested = [("%d (%s)" % (i, index_to_ngram[i])) for i in range(num_distinct_ngrams) if M_ngram_doc[i, test_docid] == True]

print()
print(f"Non-zeros in corresponding row:\n {ngrams_doc_tested}")


Positions of non-zeros in column of docid 391 of M_ngram_doc

Clean text:
 what would have happened if the japanese had not attacked pearl harbor would the us have still gotten into the war with germany

Non-zeros in corresponding row:
 ['4215 (what would have)', '4216 (would have happened)', '4217 (have happened japanese)', '4218 (happened japanese attacked)', '4219 (japanese attacked pearl)', '4220 (attacked pearl harbor)', '4221 (pearl harbor would)', '4222 (harbor would have)', '4223 (would have still)', '4224 (have still gotten)', '4225 (still gotten into)', '4226 (gotten into with)', '4227 (into with germany)']


# 5. Implement a permutation generator <font color="#2680EB">[10%]</font>

Implement the function `random_permutation(k)`, which should generate a random permutation of the array `[0, 1, 2, 3, ..., k-1]`. Tip: the function [random.shuffle](https://docs.python.org/3/library/random.html#random.shuffle) might be useful. If you want to use `range(...)`, which returns an iterator, you will need to convert the iterator to a list by using `list(range(...))`.

Remember to test your code. For instance, a permutation of 20 elements should look like this:

```
[14, 10, 0, 8, 4, 12, 5, 19, 6, 9, 15, 13, 16, 2, 17, 11, 7, 3, 18, 1]
```

Every number appears only once, and all numbers from 0 to 19 appear in the permutation.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement the function "random_permutation"**. Follow the specification above.</font>

In [ ]:
def random_permutation(k):
    # YOUR CODE HERE
    return l

We test this by applying the same permutation on two lists. The code below, which you must leave as-is,  should print both lists in the same ordering, so that *alpha* is in the same position of *a*, *beta* in the same position as *b*, and so on.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [19]:
# LEAVE AS-IS

# Permute a list according to a permutation
def permuter(original_list, permutation):
    permuted_list = []
    for index in permutation:
        permuted_list.append(original_list[index])
    return permuted_list

# Code for testing permutations

# Original lists
original_list_1 = ["1 (test1)", "2 (test2)", "3 (test3)", "4 (test4)", "5 (test5)"]
original_list_2 = ["1 (alpha)", "2 (gamma)", "3 (beta)", "4 (delta)", "5 (epsilon)"]
assert len(original_list_1) == len(original_list_2)
lists_len = len(original_list_1)

print("Original:")
print(original_list_1)
print(original_list_2)
print()

# Two permutation tests
print("Permuted (first test):")
permutation_1 = random_permutation(lists_len)
print(permuter(original_list_1, permutation_1))
print(permuter(original_list_2, permutation_1))
print()

print("Permuted (second test)")
permutation_2 = random_permutation(lists_len)
print(permuter(original_list_1, permutation_2))
print(permuter(original_list_2, permutation_2))

Original:
['1 (test1)', '2 (test2)', '3 (test3)', '4 (test4)', '5 (test5)']
['1 (alpha)', '2 (gamma)', '3 (beta)', '4 (delta)', '5 (epsilon)']

Permuted (first test):
['4 (test4)', '3 (test3)', '2 (test2)', '1 (test1)', '5 (test5)']
['4 (delta)', '3 (beta)', '2 (gamma)', '1 (alpha)', '5 (epsilon)']

Permuted (second test)
['3 (test3)', '1 (test1)', '5 (test5)', '2 (test2)', '4 (test4)']
['3 (beta)', '1 (alpha)', '5 (epsilon)', '2 (gamma)', '4 (delta)']


# 6. Compute the signature of each document <font color="#2680EB">[20%]</font>

Now comes the core of the algorithm. We will create a new matrix `M_signature_doc` having a small number of rows (the *signature size*), which will be equivalent to the number of permutations we use. The number of columns will continue being the number of documents.

First, we create the permutations and store them in an array of arrays named `permutations`, with the following code, which you should leave as-is.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [20]:
# LEAVE AS-IS

NUM_PERMUTATIONS = 5

permutations = []

# Create the permutations
for i in range(NUM_PERMUTATIONS):
    permutation = random_permutation(num_distinct_ngrams)
    permutations.append(random_permutation(num_distinct_ngrams))
    
# Visualize the permutations by printing their first 4 elements
for i in range(len(permutations)):
    permutation = permutations[i]
    print("Permutation %d: %d, %d, %d, %d, ..." % (i,
                permutation[0], permutation[1], permutation[2], permutation[3] ))

Permutation 0: 6593, 10344, 15112, 12198, ...
Permutation 1: 19558, 19417, 2126, 8395, ...
Permutation 2: 15421, 14652, 13668, 17282, ...
Permutation 3: 2394, 18992, 11632, 14829, ...
Permutation 4: 14626, 15649, 11798, 16498, ...


The following function, which you should leave as-is, finds the first n-gram that appears in a document, according to the ordering defined by a permutation.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [21]:
# LEAVE AS-IS

# Find the first ngram in a document, according to a permutation
def find_first_one(docid, permutation):

    # The instructor used AI to optimize this function (Gemini 3.6 Flash)
    
    # Extract the column docid in the order of the permutation
    matches = M_ngram_doc[permutation, docid]
    
    # Check if there are any "True" in the column
    if not matches.any():
        return -1
        
    # Function argmax returns the index of the first True value
    return permutation[np.argmax(matches)]

Now, you implement the signature construction. The matrix `M_signature_doc` should contain in row *i*, column *j*, the first ngram (the "minimum" one) that is present in a column (document), according to the order given by a permutation.

This process may take a few minutes to be completed. 

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement code to create M_signature_doc**.</font>

In [ ]:
# Create empty matrix
M_signature_doc = np.full((NUM_PERMUTATIONS, len(questions)), np.nan)

for permutation_num in range(NUM_PERMUTATIONS):
    print(f"Creating signatures for permutation {permutation_num+1} of {NUM_PERMUTATIONS}")
    permutation = permutations[permutation_num]
    
    start = timer()
    print(f"- There are {len(questions)} documents")
    # YOUR CODE HERE

# The code above should take less than 5 minutes for 10K documents

Creating signatures for permutation 1 of 5
- There are 2000 documents
- Scanning document 0 (0s)
- Scanning document 1000 (0s)
Creating signatures for permutation 2 of 5
- There are 2000 documents
- Scanning document 0 (0s)
- Scanning document 1000 (0s)
Creating signatures for permutation 3 of 5
- There are 2000 documents
- Scanning document 0 (0s)
- Scanning document 1000 (0s)
Creating signatures for permutation 4 of 5
- There are 2000 documents
- Scanning document 0 (0s)
- Scanning document 1000 (0s)
Creating signatures for permutation 5 of 5
- There are 2000 documents
- Scanning document 0 (0s)
- Scanning document 1000 (0s)


Test your code by checking the signatures of two documents that are near-duplicates,using the next code, which you should leave as-is. Being near-duplicates, we expect these should have many ngrams in common, and hence, with high probability they will have many elements in common in their signatures.

Note that your ngrams and signatures vectors might be different than what we show here, given the differences in cleaning procedures and the randomness of the permutations.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [23]:
# LEAVE AS-IS

def extract_ngrams(docid):
    return [x for x in range(num_distinct_ngrams) if M_ngram_doc[x, docid] == True]

def extract_signature(docid):
    return [M_signature_doc[x, docid] for x in range(NUM_PERMUTATIONS)]


In [25]:
# Pick two documents that are similar

TEST_DOCID_1 = 790
TEST_DOCID_2 = 791

# LEAVE AS-IS

def print_sig(texts, M_ngram_doc, M_signature_doc, i):
    print(f"Document #{i}")
    print(f"Text      : {texts[i]}")
    print(f"Clean     : {clean(texts[i])}")
    
    ngrams = extract_ngrams(i)
    signature = extract_signature(i)
    
    print(f"Ngrams    : {ngrams}")
    print(f"Signature : {signature}")
    
    return (ngrams, signature)

# Print two messages and their signatures

(ng1,sig1) = print_sig(questions, M_ngram_doc, M_signature_doc, TEST_DOCID_1 )
print()
(ng2,sig2) = print_sig(questions, M_ngram_doc, M_signature_doc, TEST_DOCID_2 )
print()

print(f"Ngrams similarity     : {jaccard_similarity(ng1,ng2)}")
print(f"Signatures similarity : {jaccard_similarity(sig1,sig2)}")

Document #790
Text      : Canada I get 10% vacation pay in lump sum once a year and it gets taxed 50% so say it was 8000 I get 4000? Is this right?
Clean     : canada i get 10 vacation pay in lump sum once a year and it gets taxed 50 so say it was 8000 i get 4000 is this right
Ngrams    : [8419, 8420, 8421, 8422, 8423, 8424, 8425, 8426, 8427]
Signature : [8425.0, 8420.0, 8422.0, 8420.0, 8419.0]

Document #791
Text      : Buying a car from different state in india and getting it registered in other state is it possible, will I have to pay taxes in both states?
Clean     : buying a car from different state in india and getting it registered in other state is it possible will i have to pay taxes in both states
Ngrams    : [8428, 8429, 8430, 8431, 8432, 8433, 8434, 8435, 8436, 8437, 8438, 8439, 8440]
Signature : [8437.0, 8438.0, 8438.0, 8439.0, 8433.0]

Ngrams similarity     : 0.0
Signatures similarity : 0.0


# 7. Compare all pairs of signatures <font color="#2680EB">[10%]</font>

Now we are ready to compare all documents by their signatures, instead of by their content.

We will consider that if two documents have *similarity == 1.0* they are a **full signature match**, and if two documents have *0.2 <= similarity < 1.0* they are a **partial signature match**. In both cases, this may mean the documents are duplicates or near duplicates.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Implement all-pairs comparison of signatures**. Keep pairs in which there is a partial or full signature match.</font>

In [ ]:
# List of possible duplicates
# Keys should be tuples (docid1, docid2)
# Values should be the similarity between the signatures of those docs.
# Only store if they are partial or full matches
is_possible_duplicate = {}
count_near_duplicates = {}

# Iterate through all documents
start = timer()
print(f"There are {len(questions)} documents")

for docid1 in range(len(questions)):
    if docid1 % 500 == 0:
        print(f"{docid1} scanned ({int(timer()-start)}s)")

    # Extract the signature of the doc1
    signature1 = extract_signature(docid1)

    # Iterate through documents with docid larger than doc1
    for docid2 in range(docid1+1, len(questions)):

        # Extract signature of doc2
        signature2 = extract_signature(docid2)

        # Compute similarity and keep in case of partial or full match
        # In that case, increase the counting of near duplicates of doc1
        # YOUR CODE HERE
print("Done")


There are 2000 documents
0 scanned (0s)
500 scanned (1s)
1000 scanned (2s)
1500 scanned (2s)
Done


The following prints random pairs of near-duplicate questions with various similarity levels.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

In [27]:
# LEAVE AS-IS

for sim in (0.2, 0.4, 0.6, 0.8, 1.0):
    print(f"Random pair with similarity {sim}")
    pairs = [pair for pair, similarity in is_possible_duplicate.items() if (abs(similarity  - sim) <= 1e-3)]
    (docid1, docid2) = random.choice(pairs)
    q1 = questions[docid1]
    q2 = questions[docid2]

    print(f"Question 1      : ({docid1}) {q1}")
    print(f"Question 2      : ({docid2}) {q2}")
    print(f"Signatures sim. : {is_possible_duplicate[(docid1,docid2)]}")    
    print(f"N-gram sim.     : {jaccard_similarity(ngrams(q1,3), ngrams(q2,3)):.2f}")
    
    print()


Random pair with similarity 0.2
Question 1      : (1463) What are good gifts for a foreign visitor to bring when they're invited to someone's home in Israel for the first time?
Question 2      : (1820) What are good gifts for a foreign visitor to bring when they're invited to someone's home in Samoa for the first time?
Signatures sim. : 0.2
N-gram sim.     : 0.60

Random pair with similarity 0.4
Question 1      : (311) What are the best job fairs near Atlanta? When and where are they held? What were your experiences at them?
Question 2      : (1704) Who are the best personal financial advisors in Roanoke for people saving for retirement? What were your experiences with them?
Signatures sim. : 0.4
N-gram sim.     : 0.09

Random pair with similarity 0.6
Question 1      : (711) How cold can the Gobi Desert get, and how do its average temperatures compare to the ones in the Colorado Plateau?
Question 2      : (1823) How cold can the Gobi Desert get, and how do its average temperatures comp

<font size="+1" color="#2680EB">**Answer the question below** regarding the differences you observe between signatures similarity and n-gram similarities.</font>

**Answer.** Why do the signature-based similarities and the n-grams similarities differ?

* (Your answer here.)


# 8. Describe near-duplicates <font color="#2680EB">[20%]</font>

**⚠️ IMPORTANT:** Now set MAX_MESSAGES to 10000 and run your code again **starting from section "4. Computing the doc-ngram matrix"**. This will scan a larger number of questions.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Print questions having many near-duplicates**. For each of them, print up to 5 of its near-duplicate elements.</font>

In [ ]:
# Keep track of which questions we have already printed
docids_in_output = {}

# Iterate through the questions with more duplicates
for top_q in sorted(count_near_duplicates, key=count_near_duplicates.get, reverse=True):

    # Skip questions that have already been printed
    # as near-duplicates of another question
    if top_q in docids_in_output:
        continue
    docids_in_output[top_q] = True
    
    # Skip questions that have less than 5 near-duplicates
    if count_near_duplicates[top_q] < 5:
        continue
    
    # Print the question
    print()
    print(f"Question {top_q} has many duplicates")
    print(f"Text: {questions[top_q]}")
    
    # Print up to 3 near-duplicates of that question
    # Keep track of how many you "printed" out of how many "existed"
    num_printed = 0
    num_existed = 0
    
    # YOUR CODE HERE      
    
    print(f"Printed {num_printed}/{num_existed} near-duplicates")



Question 59 has many duplicates
Text: What hotel in Dehradun Hill-station would be safe for unmarried couples, without the harassment of police, hotel staff, and moral police?
Near-duplicate : What hotel in Pelling Hill-station would be safe for unmarried couples, without the harassment of police, hotel staff, and moral police?
Sig similarity : 0.5
Near-duplicate : What hotel in Lachung Hill-station would be safe for unmarried couples, without the harassment of police, hotel staff, and moral police?
Sig similarity : 0.4
Near-duplicate : What hotel in Saharanpur would be safe for unmarried couples, without the harassment of police, hotel staff, and moral police?
Sig similarity : 0.4
Printed 3/48 near-duplicates

Question 58 has many duplicates
Text: What hotel in Shimla Hill-station would be safe for unmarried couples, without the harassment of police, hotel staff, and moral police?
Near-duplicate : What hotel in Dehradun Hill-station would be safe for unmarried couples, without the ha

There seems to be three kinds of near-duplicates:

* Automatically-generated repetitions of a question changing some part of it. These usually have many repetitions.
* Actual duplicates, i.e., a question appearing twice.
* Near-duplicates, i.e., questions asked in two slightly different ways.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+1" color="#2680EB">**Answer the questions below** regarding the questions that have many near-duplicates.</font>

**Answer.** 

Provide two examples of questions having many near-duplicates that seem to be machine generated:

* (Your answer here.)

Why do you think someone would issue that many near-duplicate questions? Provide a plausible conjecture for one of them.

* (Your answer here.)

Write a question that seems to have been legitimately asked in two very similar ways in the platform. This question may not have many near-duplicates. Copy-paste two ways in which this question was written. Highlight in boldface the differences between the two formulations (perhaps a single word).

* (Your answer here.)



# DELIVER (individually)

Please set `MAX_MESSAGES` back to *2000* before delivering 🙏🏾 There is no need for you to re-run your code, it just helps the reviewing.

Remember to read the section on "delivering your code" in the [course evaluation guidelines](https://github.com/chatox/data-mining-course/blob/master/upf/upf-evaluation.md).

Deliver this notebook, renamed as `PSxx_Unnnnnn_YourName.ipynb` where:
* *PSxx* is the practice number.
* *Unnnnnn* is your UPF id number.
* *YourName* is your name.

## Extra points available

For more learning and extra points, choose **one** of these two activities:

* see what happens with smaller and larger ngram sizes (2-grams, 4-grams, 5-grams)
* see what happens with more permutations and fewer permutations (3 permutations, 10 permutations)

Pick one of the two above and compare them in terms of the efficiency (speed) and effectiveness (accuracy). You can include plots for efficiency, and examples for effectiveness.

**Note:** if you go for the extra points, add an additional section ``# <font color="#2680EB">Additional results: (variation of ngram size | variation of number of permutations)</font>`` at the end of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>

<font size="+2" color="#00A365">I hereby declare that I completed this practice myself, that I have not copied it from others, or from external sources without citing them, and that I have followed the generative AI instructions stated above.</font>